# Glow25 — Purchase Funnel Analysis

Glow25 sells collagen products DTC across Germany, Netherlands, Belgium, and France. They're growing fast but nobody has a clear picture of where the funnel breaks differently by country, device, or channel. This notebook is my attempt to map that out properly — not just the headline CVR, but the stage-by-stage picture that tells you *where* to fix things, not just *that* something is broken.

Data comes from an e-commerce clickstream dataset (view → cart → purchase events), enriched with simulated Glow25 market dimensions.

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import sqlite3
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

df = pd.read_csv('../data/processed/events_clean.csv', parse_dates=['event_time'])
print(f"Loaded {len(df):,} rows")
df.head()

In [ ]:
# sanity checks before doing any analysis
print("Event types:", df['event_type'].value_counts().to_dict())
print("Countries:", df['country'].value_counts().to_dict())
print("Devices:", df['device'].value_counts().to_dict())
print("Channels:", df['channel'].value_counts().to_dict())
print("\nNull counts:")
print(df[['event_time', 'user_id', 'event_type', 'country', 'device', 'channel']].isnull().sum())
print(f"\nDate range: {df['event_time'].min()} → {df['event_time'].max()}")

## 2. Funnel Construction

The funnel here is: view → cart → purchase. Each user can appear in multiple stages, so I'm collapsing to per-user flags first — did they view, did they cart, did they buy. Otherwise heavy browsers inflate the view count and make CVR look worse than it is.

In [ ]:
# collapse to one row per user with boolean flags per stage
user_events = df.groupby('user_id')['event_type'].apply(set).reset_index()
user_events.columns = ['user_id', 'events_set']

user_events['viewed']    = user_events['events_set'].apply(lambda s: 'view' in s)
user_events['carted']    = user_events['events_set'].apply(lambda s: 'cart' in s)
user_events['purchased'] = user_events['events_set'].apply(lambda s: 'purchase' in s)

viewers    = user_events['viewed'].sum()
carted     = user_events['carted'].sum()
purchasers = user_events['purchased'].sum()

v2c_rate = carted / viewers
c2p_rate = purchasers / carted
overall  = purchasers / viewers

print(f"Viewers:    {viewers:,}")
print(f"Carted:     {carted:,}   ({v2c_rate:.1%} of viewers)")
print(f"Purchased:  {purchasers:,}   ({c2p_rate:.1%} of carted, {overall:.1%} overall CVR)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

stages = ['Viewed', 'Added to Cart', 'Purchased']
counts = [viewers, carted, purchasers]
colors = ['#D4537E', '#a83460', '#7a2445']

bars = ax.barh(stages[::-1], counts[::-1], color=colors[::-1])

for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
            f'{count:,}', va='center', fontsize=10)

ax.set_xlabel('Unique Users')
ax.set_title('Purchase Funnel — Overall')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 3. Country-Level Conversion Analysis

This is the main question I wanted to answer. Germany is the biggest market so even a small CVR difference there has big revenue impact. I want to see both funnel steps separately — view→cart is a product/content problem, cart→purchase is a checkout/trust problem.

In [ ]:
# merge country back onto user-level data
# each user gets the country they appear most in (mode), since country is per-event
user_country = df.groupby('user_id')['country'].agg(lambda x: x.mode()[0]).reset_index()
user_events = user_events.merge(user_country, on='user_id')

country_funnel = user_events.groupby('country').agg(
    viewers=('viewed', 'sum'),
    carted=('carted', 'sum'),
    purchasers=('purchased', 'sum'),
).reset_index()

country_funnel['v2c_pct'] = (country_funnel['carted'] / country_funnel['viewers'] * 100).round(2)
country_funnel['c2p_pct'] = (country_funnel['purchasers'] / country_funnel['carted'] * 100).round(2)
country_funnel['overall_cvr'] = (country_funnel['purchasers'] / country_funnel['viewers'] * 100).round(2)

country_funnel.sort_values('overall_cvr', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = [
    ('v2c_pct',     'View → Cart %',          '#D4537E'),
    ('c2p_pct',     'Cart → Purchase %',       '#a83460'),
    ('overall_cvr', 'Overall CVR %',           '#7a2445'),
]

for ax, (col, label, color) in zip(axes, metrics):
    data = country_funnel.sort_values(col, ascending=True)
    ax.barh(data['country'], data[col], color=color)
    ax.set_xlabel(label)
    ax.set_title(label)
    for i, (val, country) in enumerate(zip(data[col], data['country'])):
        ax.text(val * 1.02, i, f'{val}%', va='center', fontsize=9)

plt.suptitle('Funnel Conversion by Country', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Device Breakdown

In [ ]:
user_device = df.groupby('user_id')['device'].agg(lambda x: x.mode()[0]).reset_index()
user_events_dev = user_events.merge(user_device, on='user_id')

device_funnel = user_events_dev.groupby('device').agg(
    viewers=('viewed', 'sum'),
    carted=('carted', 'sum'),
    purchasers=('purchased', 'sum'),
).reset_index()

device_funnel['v2c_pct']     = (device_funnel['carted'] / device_funnel['viewers'] * 100).round(2)
device_funnel['c2p_pct']     = (device_funnel['purchasers'] / device_funnel['carted'] * 100).round(2)
device_funnel['overall_cvr'] = (device_funnel['purchasers'] / device_funnel['viewers'] * 100).round(2)

device_funnel.sort_values('overall_cvr', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(device_funnel))
width = 0.25

ax.bar(x - width, device_funnel['v2c_pct'],     width, label='View→Cart %',     color='#D4537E')
ax.bar(x,         device_funnel['c2p_pct'],     width, label='Cart→Purchase %', color='#a83460')
ax.bar(x + width, device_funnel['overall_cvr'], width, label='Overall CVR %',   color='#7a2445')

ax.set_xticks(x)
ax.set_xticklabels(device_funnel['device'])
ax.set_ylabel('Rate (%)')
ax.set_title('Funnel Rates by Device')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Channel Analysis

In [ ]:
user_channel = df.groupby('user_id')['channel'].agg(lambda x: x.mode()[0]).reset_index()
user_events_ch = user_events.merge(user_channel, on='user_id')

channel_funnel = user_events_ch.groupby('channel').agg(
    viewers=('viewed', 'sum'),
    carted=('carted', 'sum'),
    purchasers=('purchased', 'sum'),
).reset_index()

channel_funnel['overall_cvr'] = (channel_funnel['purchasers'] / channel_funnel['viewers'] * 100).round(2)
channel_funnel.sort_values('overall_cvr', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

data = channel_funnel.sort_values('overall_cvr', ascending=True)
colors = sns.color_palette('RdPu', n_colors=len(data))
ax.barh(data['channel'], data['overall_cvr'], color=colors)
ax.set_xlabel('Overall CVR (%)')
ax.set_title('Purchase CVR by Acquisition Channel')

for i, val in enumerate(data['overall_cvr']):
    ax.text(val + 0.02, i, f'{val}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Cohort Analysis — Channel CVR Comparison

The question I want to answer here: do email users convert at a structurally different rate than paid social users? If so, that's not about the channel budget, it's about the audience quality and intent level when they arrive.

In [ ]:
# TODO: add time-to-purchase distribution per channel — would show whether email
# users convert faster (same session) or slower (return visits)

# channel × country CVR heatmap
pivot = user_events_ch.merge(user_country, on='user_id')

ch_country = pivot.groupby(['channel', 'country']).agg(
    viewers=('viewed', 'sum'),
    purchasers=('purchased', 'sum'),
).reset_index()

ch_country['cvr'] = (ch_country['purchasers'] / ch_country['viewers'] * 100).round(2)

heatmap_data = ch_country.pivot(index='channel', columns='country', values='cvr')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heatmap_data, annot=True, fmt='.1f', cmap='RdPu',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'CVR (%)'}
)
ax.set_title('CVR (%) by Channel × Country')
plt.tight_layout()
plt.show()

In [ ]:
# cohort by acquisition channel — compare conversion rates
channel_cvr_summary = (
    user_events_ch
    .groupby('channel')
    .agg(
        users=('user_id', 'count'),
        converters=('purchased', 'sum')
    )
    .assign(cvr=lambda x: (x['converters'] / x['users'] * 100).round(2))
    .sort_values('cvr', ascending=False)
)

print(channel_cvr_summary.to_string())

## 7. Key Findings

Writing these as I'd write them in a Slack message to the head of growth — specific, actionable, no fluff.

**Funnel structure.** The biggest drop is at view→cart, which is typical for DTC. Most users browse but don't add to cart. The cart→purchase step is where the country differences emerge more clearly — that's a checkout problem, not a product problem.

**Country differences.** France and Belgium show lower overall CVR than DE and NL. This is likely a combination of lower brand awareness (smaller markets, less word-of-mouth) and potentially payment method mismatches — Bancontact in BE, Carte Bancaire trust issues in FR. Worth investigating before scaling paid spend in those markets.

**Device gap.** Mobile converts lower than desktop, as expected. The question is whether the gap is larger than industry norms (~1.5-2× is typical). If it's 3×+, the mobile checkout UX needs work before you scale paid social (which is heavily mobile-dominant).

**Channel quality.** Email has the highest purchase intent — these are existing customers or newsletter subscribers who already trust the brand. Paid social drives volume but at lower CVR. That's not a problem as long as the CAC math works; the risk is optimising for CVR and cutting paid social budget prematurely.

**What I'd test next.** Simplify the mobile checkout (reduce form fields, add Apple/Google Pay), A/B test localised payment methods in BE and FR, and set up a cart abandonment email flow for users who don't purchase within 2 hours of carting.

In [ ]:
# save to SQLite for the SQL analysis layer
conn = sqlite3.connect('../data/glow25.db')
df.to_sql('events', conn, if_exists='replace', index=False)
conn.close()
print("Saved events table to data/glow25.db")